# Elections au Conseil national — Exploration des données brutes (OFS px-x-1702020000_105)

Suffrages des partis et force des partis depuis 1971 : districts et communes.

Des données existent pour les années 1971, 1975, 1979, 1983, 1987, 1991, 1995, 1999, 2003, 2007, 2011, 2015, 2019 et 2023.
Pour notre analyse, nous nous concentrons sur les années postérieures à 2010 : **2011, 2015, 2019, 2023**.

- Collecte : `python scripts/collect_elections_conseil_national.py` → `data/raw/elections_conseil_national_brut.csv`
- Documentation de la source : `docs/datasets/elections_px_105/README.md`
- Ce notebook **ne lit que `data/raw/`** et ne modifie rien.

# 1. Imports

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Chargement des données brutes

In [15]:
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Racine du dépôt introuvable (aucun dossier .git en remontant depuis le dossier courant).")

RAW_CSV = PROJECT_ROOT / "data" / "raw" / "elections_conseil_national_brut.csv"
if not RAW_CSV.exists():
    raise FileNotFoundError(f"{RAW_CSV} introuvable : lancez d'abord python scripts/collect_elections_conseil_national.py")

df_elec = pd.read_csv(RAW_CSV)
df_elec.shape

(466600, 5)

## 3. Vue d'ensemble des données brutes

In [16]:
col_valeur = "value"
col_geo, col_annee, col_parti, col_indic = df_elec.columns[:4]
df_elec[col_annee] = df_elec[col_annee].astype(str)

print("Colonne géographique :", col_geo)
print("Colonne année        :", col_annee)
print("Colonne parti        :", col_parti)
print("Colonne indicateur   :", col_indic)
print("\nIndicateurs présents :", df_elec[col_indic].unique().tolist())
print("\nExemples de géographie :", df_elec[col_geo].drop_duplicates().head(8).tolist())

Colonne géographique : District (>>) / Commune (......)
Colonne année        : Année
Colonne parti        : Parti
Colonne indicateur   : Résultats

Indicateurs présents : ['Suffrages des partis', 'Force des partis en %']

Exemples de géographie : ['>> Bezirk Affoltern', '......Aeugst am Albis', '......Affoltern am Albis', '......Bonstetten', '......Hausen am Albis', '......Hedingen', '......Kappel am Albis', '......Knonau']


In [17]:
df_elec.head(10)

,District (>>) / Commune (......),Année,Parti,Résultats,value
0,>> Bezirk Affoltern,2023,PLR,Suffrages des partis,81687.00
1,>> Bezirk Affoltern,2023,PLR,Force des partis en %,12.34
2,>> Bezirk Affoltern,2023,Centre,Suffrages des partis,53388.00
3,>> Bezirk Affoltern,2023,Centre,Force des partis en %,8.07
4,>> Bezirk Affoltern,2023,PS,Suffrages des partis,102764.00
5,>> Bezirk Affoltern,2023,PS,Force des partis en %,15.53
6,>> Bezirk Affoltern,2023,UDC,Suffrages des partis,217081.00
7,>> Bezirk Affoltern,2023,UDC,Force des partis en %,32.80
8,>> Bezirk Affoltern,2023,PL,Suffrages des partis,NaN
9,>> Bezirk Affoltern,2023,PL,Force des partis en %,NaN


### Structure géographique

Cette table ne contient que deux niveaux : les **districts** (préfixe `>>`) et les **communes** (préfixe `......`).
Elle ne contient ni la Suisse ni les cantons. Le référentiel communal est celui du **01.07.2023**.

Des **entités spéciales** (votes non attribués, Suisses de l'étranger, vote par correspondance, communes
sans bureau de vote propre) apparaissent aux deux niveaux. Ce ne sont pas de vraies communes ou de vrais
districts : elles sont repérées par leur nom (`Anderes`, `Korrespondenzweg`, `autres`, `CH de l'étranger`, `Ausland`).

In [26]:
speciales = r"Anderes|Korrespondenzweg|autres|CH de l'étranger|Ausland"

def niveau_geo(s):
    return np.select(
        [s.str.startswith(">>"), s.str.startswith("......")],
        ["district", "commune"],
        default="autre",
    )

geo = df_elec[[col_geo]].drop_duplicates().copy()
geo["niveau"] = niveau_geo(geo[col_geo])
geo["entite_speciale"] = geo[col_geo].str.contains(speciales)
geo.groupby(["niveau", "entite_speciale"]).size()

niveau    entite_speciale
commune   False              2138
          True                 25
district  False               145
          True                 25
dtype: int64

## 4. Exploration et contrôle qualité

### 4.1 Couverture temporelle, géographique et par parti

In [19]:
annees = sorted(df_elec[col_annee].unique())
partis = sorted(df_elec[col_parti].unique())

print(f"{len(annees)} élections : {annees}")
print(f"{len(geo)} unités géographiques distinctes")
print(f"{len(partis)} partis : {partis}")
print(f"Lignes attendues : {len(geo)} × {len(annees)} × {len(partis)} × {df_elec[col_indic].nunique()} = "
      f"{len(geo) * len(annees) * len(partis) * df_elec[col_indic].nunique():,} (obtenues : {len(df_elec):,})")

4 élections : ['2011', '2015', '2019', '2023']
2333 unités géographiques distinctes
25 partis : ['AVF', 'AdI', 'Autres', 'Centre', 'DS', 'Lega', 'MCR', 'PBD', 'PCS', 'PDC', 'PEV', 'PL', 'PLR', 'POCH', 'PS', 'PSA', 'PSL', 'PST', 'PVL', 'Rép.', 'Sol.', 'Sép.', 'UDC', 'UDF', 'Verts']
Lignes attendues : 2333 × 4 × 25 × 2 = 466,600 (obtenues : 466,600)


### 4.2 Valeurs manquantes

Dans ce tableau, une valeur manquante signifie en général que le **parti n'a pas présenté de liste**
dans l'unité concernée (manquant structurel), pas qu'une donnée est perdue. Il faut donc distinguer
`NaN` (pas de liste / pas de donnée) de `0` (liste présente, aucun suffrage) lors du traitement.

In [20]:
print(f"Valeurs manquantes : {df_elec[col_valeur].isna().sum():,} ({df_elec[col_valeur].isna().mean():.1%})")
print("\nPar année :")
print(df_elec.groupby(col_annee)[col_valeur].apply(lambda s: s.isna().mean()).round(3))
print("\nPar indicateur :")
print(df_elec.groupby(col_indic)[col_valeur].apply(lambda s: s.isna().mean()).round(3))

Valeurs manquantes : 280,540 (60.1%)

Par année :
Année
2011    0.580
2015    0.603
2019    0.593
2023    0.629
Name: value, dtype: float64

Par indicateur :
Résultats
Force des partis en %    0.601
Suffrages des partis     0.601
Name: value, dtype: float64


### 4.3 Présence des partis par élection

Nombre de **communes** avec une force du parti renseignée, par parti et par année. Cela met en évidence
les partis sans aucune donnée sur la période et les fusions de partis (ex. PDC + PBD → Le Centre en 2021 ;
PRD/PLS → PLR ; PL dans le canton de Vaud en 2011).

In [21]:
masque_force = df_elec[col_indic].str.contains("Force", case=False)
masque_commune = df_elec[col_geo].str.startswith("......")

force_communes = df_elec[masque_force & masque_commune]
presence = (
    force_communes.dropna(subset=[col_valeur])
    .groupby([col_parti, col_annee]).size()
    .unstack(fill_value=0)
    .reindex(index=partis, columns=annees, fill_value=0)
)
presence

Année,2011,2015,2019,2023
Parti,,,,
AVF,639,187,187,161
AdI,0,0,0,0
Autres,1557,1958,1831,1975
Centre,0,0,0,2136
DS,1157,949,1057,650
Lega,106,106,106,106
MCR,347,46,46,46
PBD,1835,1730,1623,0
PCS,409,301,301,127


In [22]:
sans_donnees = presence.index[(presence == 0).all(axis=1)].tolist()
print("Partis sans aucune donnée communale sur la période :", sans_donnees)

Partis sans aucune donnée communale sur la période : ['AdI', 'POCH', 'PSA', 'PSL', 'Rép.', 'Sép.']


### 4.4 Cohérence : somme des forces des partis par commune

Pour chaque commune et chaque élection, la somme des « forces des partis en % » doit valoir environ 100 %.

In [23]:
sommes = (
    force_communes.groupby([col_geo, col_annee])[col_valeur]
    .sum(min_count=1)
    .dropna()
)
print(sommes.describe().round(2))
print(f"\nCommunes-années avec une somme hors de [99, 101] : {((sommes < 99) | (sommes > 101)).sum()} sur {len(sommes):,}")

count    8565.00
mean      100.00
std         0.01
min        99.96
25%        99.99
50%       100.00
75%       100.01
max       100.04
Name: value, dtype: float64

Communes-années avec une somme hors de [99, 101] : 0 sur 8,565


### 4.5 Communes sans aucune donnée pour une élection

Cas typique : communes sans bureau de vote propre (« Urne commune »), dont les résultats sont comptés ailleurs.

In [27]:
tout = force_communes.groupby([col_geo, col_annee])[col_valeur].apply(lambda s: s.notna().sum())
sans_donnee = tout[tout == 0].reset_index()
sans_donnee["type"] = np.where(sans_donnee[col_geo].str.contains(speciales), "entité spéciale", "commune")

print(f"{len(sans_donnee)} couples commune-année sans aucune donnée")
print(sans_donnee["type"].value_counts())
sans_donnee[sans_donnee["type"] == "commune"]

87 couples commune-année sans aucune donnée
type
entité spéciale    54
commune            33
Name: count, dtype: int64


,District (>>) / Commune (......),Année,value,type
12,......Deisswil bei Münchenbuchsee,2011,0,commune
13,......Deisswil bei Münchenbuchsee,2015,0,commune
14,......Deisswil bei Münchenbuchsee,2019,0,commune
15,......Deisswil bei Münchenbuchsee,2023,0,commune
29,......Hellsau,2011,0,commune
30,......Hellsau,2015,0,commune
31,......Hellsau,2019,0,commune
32,......Hellsau,2023,0,commune
33,......Jaberg,2011,0,commune
34,......Jaberg,2015,0,commune


### 4.6 Doublons

In [25]:
doublons = df_elec.duplicated(subset=[col_geo, col_annee, col_parti, col_indic], keep=False)
print(f"{doublons.sum()} lignes dupliquées sur {len(df_elec):,} (clé : géo × année × parti × indicateur)")

0 lignes dupliquées sur 466,600 (clé : géo × année × parti × indicateur)
